In [6]:
import pandas as pd
import numpy as np

def extend_cashflow_csv(file_path, months_to_add=60, seed=123):
    np.random.seed(seed)

    # Load existing file
    df = pd.read_csv(file_path)

    # Ensure correct order
    df["Date"] = pd.to_datetime(df["Year"].astype(str) + "-" + df["Month"])
    df = df.sort_values("Date").reset_index(drop=True)

    # Get last known values
    last_row = df.iloc[-1]
    last_date = last_row["Date"]
    last_closing_balance = last_row["Closing Balance"]

    new_rows = []

    current_balance = last_closing_balance

    for i in range(months_to_add):

        next_date = last_date + pd.DateOffset(months=1)

        # Trend growth based on previous Operating CF
        base = last_row["Operating Cash Flow"] * np.random.uniform(0.98, 1.08)

        # Seasonality
        seasonal_factor = 1 + 0.08 * np.sin(2 * np.pi * next_date.month / 12)

        # Volatility
        noise = np.random.normal(0, 0.06 * base)

        operating_cf = base * seasonal_factor + noise
        investing_cf = -0.30 * base + np.random.normal(0, 0.05 * base)
        financing_cf = np.random.normal(0, 0.15 * base)

        net_cf = operating_cf + investing_cf + financing_cf

        current_balance += net_cf

        new_rows.append({
            "Year": next_date.year,
            "Month": next_date.month,
            "Operating Cash Flow": round(operating_cf, 0),
            "Investing Cash Flow": round(investing_cf, 0),
            "Financing Cash Flow": round(financing_cf, 0),
            "Net Cash Flow": round(net_cf, 0),
            "Closing Balance": round(current_balance, 0)
        })

        # Update references
        last_row = {
            "Operating Cash Flow": operating_cf
        }
        last_date = next_date

    new_df = pd.DataFrame(new_rows)

    # Combine old + new
    updated_df = pd.concat([df.drop(columns=["Date"]), new_df], ignore_index=True)

    # Save updated file
    updated_df.to_csv("csv1.csv", index=False)

    print("Dataset extended successfully!")
    print(f"New total rows: {len(updated_df)}")

    return updated_df


# Example usage:
extend_cashflow_csv("cashflow.csv", months_to_add=120)

Dataset extended successfully!
New total rows: 180


C:\Users\Admin\AppData\Local\Temp\ipykernel_15820\1478389030.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Year"].astype(str) + "-" + df["Month"])


,Year,Month,Opening Balance,Operating Cash Flow,Investing Cash Flow,Financing Cash Flow,Net Cash Flow,Closing Balance
0,2020,Jan,500000.0,100000.0,-20000.0,-15000.0,65000.0,565000.0
1,2020,Feb,565000.0,105000.0,-15000.0,-15000.0,75000.0,640000.0
2,2020,Mar,640000.0,110000.0,-50000.0,-15000.0,45000.0,685000.0
3,2020,Apr,685000.0,115000.0,-20000.0,-15000.0,80000.0,765000.0
4,2020,May,765000.0,120000.0,-15000.0,-15000.0,90000.0,855000.0
...,...,...,...,...,...,...,...,...
175,2034,8,NaN,17858733.0,-5749496.0,-3211132.0,8898105.0,558908285.0
176,2034,9,NaN,15983132.0,-4898480.0,7299256.0,18383908.0,577292193.0
177,2034,10,NaN,17368090.0,-4059027.0,-1389007.0,11920057.0,589212250.0
178,2034,11,NaN,17446366.0,-5925971.0,551721.0,12072116.0,601284366.0
